In [ ]:
#-------------------------------------------------------------------------------
# Name:        03_convert_Masks
# Purpose:     Rasterize the vecor input mask so that they are 256x256px
#
# Author:      Gijs van den Dool & Meggison Oritsemisa
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/1dgFGkQ0Orjt8SbPtpOvPNb1A9y4Odeng?usp=drive_link

# Vector Masks:
# Output from the selection of IA areas in a 2560x2560 bounding box

# Raster Masks:
# Input for the modelling, as a direct conversion from vector to raster


In [ ]:
# check packages installed:
packages = !pip list

if "mapclassify" in packages:
  print("package available")
else:
  # for visualisation of a geopandas dataframe
  !pip install mapclassify -q


if "rasterio" in packages:
  print("package available")
else:
  # for visualisation of a geopandas dataframe
  !pip install rasterio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 38.7 MB/s eta 0:00:00


In [ ]:
import os

import geemap

import json

from shapely.geometry import Point,LineString, Polygon

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import geopandas as gpd

from matplotlib import pyplot as plt

/Users/misanmeggison/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
project_root = "/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_IA" # Change Project folder
task_folder = "02_WorkingFiles"
file_name = "mask_IA_60.geojson" # might be different for other areas

file_path = os.path.join(project_root, task_folder, file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
    print("File exists.")
else:
    print("File does not exist or is not accessible.")

File exists.


In [ ]:
# Create output folder:
task_folder = "03_Output_Data"
out_folder = "01_masks/"
out_folder = os.path.join(project_root, task_folder, out_folder)

if not os.path.exists(out_folder):
    os.makedirs(out_folder)
    print("Folder created successfully.")
else:
    print("Folder already exists.")

Folder already exists.


In [ ]:
gdf_ISL_Mask = gpd.read_file(file_path)

print(len(gdf_ISL_Mask)) # total lines in the AoI
print(gdf_ISL_Mask.crs)  # current projection
gdf_ISL_Mask.head(3)

14
EPSG:4326


,GID,AoI_id_2,mask,geometry
0,GID_IA_60_0_1,IA_60,1,"POLYGON ((15.25384 -0.74808, 15.25417 -0.74778..."
1,GID_IA_60_0_2,IA_60,1,"POLYGON ((15.26238 -0.75669, 15.26104 -0.75537..."
2,GID_IA_60_1_0,IA_60,1,"MULTIPOLYGON (((15.27051 -0.71095, 15.28121 -0..."


In [ ]:
#create an unique list of masks:
lst_GID = gdf_ISL_Mask.GID.unique().tolist()

In [ ]:
# pixels by patch: IA
shape = 256, 256

In [ ]:
# ToDo: add CRS to the exported tiff
# crs="+proj=latlong",

# creating the masks:
for i in range(len(lst_GID)):
  str_image_name = lst_GID[i]
  image_path = out_folder + str_image_name + '.tif'

  # set the patch extent
  rslt_df= gdf_ISL_Mask.loc[(gdf_ISL_Mask['GID'] == lst_GID[i])]
  rslt_df

  # set the mask
  mask_df = gdf_ISL_Mask.loc[(gdf_ISL_Mask['GID'] == lst_GID[i]) & (gdf_ISL_Mask['mask'] == 1) ]
  mask_df

  # The projection is kept in the original projection, but here it is possible to
  # change if an other projection is needed:
  # rslt_df = rslt_df.to_crs("32632")
  # rslt_df.crs

  # mask_df = mask_df.to_crs("32632")
  # mask_df.crs

  # set the transformation:
  transform = rasterio.transform.from_bounds(*rslt_df['geometry'].total_bounds, *shape)
  transform

  # create the mask:
  rasterize_mask = rasterize(
      [(shape, 1) for shape in mask_df['geometry']],
      out_shape=shape,
      transform=transform,
      fill=0,
      all_touched=True,
      dtype=rasterio.uint8,
      default_value=1)

  # save the raster:
  with rasterio.open(
      image_path, 'w',
      driver='GTiff',
      dtype=rasterio.uint8,
      count=1,
      width=shape[0],
      height=shape[1],
      transform=transform
  ) as dst:
      dst.write(rasterize_mask, indexes=1)